# Validación del pipeline (Parcial MCD-402)

Este notebook ejecuta **las mismas tareas que el DAG de Airflow** (`src/pipeline.py`), una tras otra, sin Docker:

`validar_datos → preprocesar_datos → optimizar_hiperparametros → entrenar_modelos → backtesting → seleccionar_campeon → registrar_experimentos → generar_reporte`

Sirve para revisar el flujo en Google Colab o en tu máquina. Como en Colab no hay servidor de MLflow, los runs se guardan en `mlflow_local.db` (respaldo definido en `config.yaml → tracking.tracking_uri_respaldo`).

In [1]:
# Solo en Google Colab: montar Drive (en tu maquina, salta esta celda)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

In [2]:
import os, sys

if EN_COLAB:
    PROJECT_ROOT = '/content/drive/MyDrive/MLOps/automl_venta_financiera'  # <-- editar
else:
    PROJECT_ROOT = os.path.abspath('..')  # el notebook esta en notebooks/

DATASET_PATH = 'data/input'   # o 'data/muestra' para una prueba rapida
sys.path.append(PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
os.environ['PROYECTO_DIR'] = PROJECT_ROOT
print('Proyecto en:', PROJECT_ROOT)

Proyecto en: C:\Users\Pao\Desktop\p3_colaboracion\mlops_tarea02


In [3]:
# Librerias (en Colab). En tu maquina: pip install -r requirements.txt
if EN_COLAB:
    !pip install -q catboost lightgbm optuna mlflow psutil PyYAML

## Evidencias de la corrida oficial

Las salidas de este notebook corresponden a la corrida oficial `data/output/2026-10-12_14-11-08` (campeon `LightGBM_hpo`), ejecutada por el DAG de Airflow en Docker. Se leen sus resultados sin volver a entrenar, para que coincidan con el README. Para correr el pipeline completo desde cero, cambiar `EJECUTAR_PIPELINE = True`.


In [4]:
# EJECUTAR_PIPELINE = True  -> corre las 8 tareas del DAG desde cero (~15 min, necesita data/input).
# EJECUTAR_PIPELINE = False -> muestra la corrida oficial ejecutada por el DAG de Airflow (sin reentrenar).
EJECUTAR_PIPELINE = False
CORRIDA_OFICIAL = 'data/output/2026-10-12_14-11-08'

if EJECUTAR_PIPELINE:
    from datetime import datetime
    from src import pipeline

    dag_run_id = 'notebook__' + datetime.now().strftime('%Y-%m-%dT%H:%M:%S')
    carpeta = None
    for tarea in pipeline.ORDEN_TAREAS:
        salida = pipeline.ejecutar_tarea(tarea, dag_run_id, carpeta_corrida=carpeta,
                                         overrides={'input_path': DATASET_PATH})
        carpeta = salida['carpeta_corrida']
else:
    carpeta = CORRIDA_OFICIAL
print('Corrida:', carpeta)

Corrida: data/output/2026-10-12_14-11-08


In [5]:
import json
import pandas as pd

with open(os.path.join(carpeta, 'results_summary.json'), encoding='utf-8') as f:
    resumen = json.load(f)

print('Campeon:', resumen['campeon'], '| estado:', resumen['seleccion']['estado'])
print(resumen['seleccion']['motivo'])
pd.DataFrame(resumen['tabla_modelos'])

Campeon: LightGBM_hpo | estado: OK
LightGBM_hpo: lift promedio en backtesting 4.9455 (mejor del grupo: 4.9796); 14 modelo(s) compitieron en la etapa final. Gana por costo: esta a menos de 2.0% del lift de CatBoost_cfg2 y entrena en 17.49s vs 58.32s.


,modelo,familia,hardware,auc_train,auc_test,decay_pct,lift_test,ks_test,pr_auc_test,bt_auc_prom,bt_lift_prom,bt_decay_prom,bt_drift_alert,tiempo_s,tamano_modelo_mb,es_campeon,descartado_por
0,CatBoost_cfg2,base_arboles_ensamble,cpu,0.8588,0.8499,1.03,5.1147,0.5473,0.1764,0.8471,4.9796,1.35,False,58.32,0.439,False,None
1,XGBoost_hpo,base_arboles_ensamble,cpu,0.8654,0.8517,1.58,5.1424,0.5484,0.1756,0.8484,4.9719,1.96,False,17.68,0.451,False,None
2,CatBoost_cfg1,base_arboles_ensamble,cpu,0.8533,0.8486,0.55,5.1101,0.5446,0.1752,0.8450,4.9653,0.97,False,29.16,0.125,False,None
3,LightGBM_hpo,base_arboles_ensamble,cpu,0.8679,0.8503,2.02,5.1206,0.5464,0.1742,0.8476,4.9455,2.34,False,17.49,0.522,True,None
4,AdaBoost_cfg2,arboles_ensamble,cpu,0.8485,0.8453,0.37,4.9689,0.5390,0.1660,0.8425,4.8604,0.71,False,212.61,0.186,False,None
5,Random_Forest_cfg1,arboles_ensamble,cpu,0.8683,0.8398,3.28,4.9557,0.5280,0.1674,0.8354,4.8000,3.79,True,120.89,6.845,False,None
6,AdaBoost_cfg1,arboles_ensamble,cpu,0.8473,0.8435,0.45,4.9010,0.5348,0.1637,0.8400,4.7810,0.86,False,93.20,0.064,False,None
7,Regresion_Logistica_hpo,lineal,cpu,0.8313,0.8308,0.07,4.7796,0.5091,0.1561,0.8265,4.5661,0.58,False,1.38,0.003,False,None
8,MLP_cfg1,redes_neuronales,cpu,0.8404,0.8334,0.83,4.7724,0.5191,0.1558,0.8285,4.5446,1.41,False,17.88,0.143,False,None
9,MLP_cfg2,redes_neuronales,cpu,0.8652,0.8375,3.20,4.7908,0.5241,0.1551,0.8317,4.5309,3.87,True,21.38,0.185,False,None


In [6]:
# Backtesting del campeon (meses posteriores al corte, sin reentrenar)
pd.DataFrame(resumen['backtesting_campeon']['waves'])

,partition,p_fecinformacion,auc,decay_pct,n_obs,psi,psi_interpretacion,lift,ks,pr_auc
0,p7,20220901,0.8342,3.88,244666,0.0094,sin cambio significativo,4.5920,0.5175,0.1518
1,p8,20221001,0.8480,2.30,240726,0.0073,sin cambio significativo,4.9259,0.5366,0.2684
2,p9,20221101,0.8503,2.03,263753,0.0028,sin cambio significativo,4.9619,0.5520,0.1723
3,p10,20221201,0.8580,1.14,317378,0.0117,sin cambio significativo,5.3024,0.5621,0.1532


In [7]:
# Logs de computo por modelo y por tarea
display(pd.DataFrame(resumen['compute_logs']['modelos']).T[['duracion_s', 'cpu_pct_prom', 'cpu_pct_max',
                                                           'ram_mb_prom', 'ram_mb_max', 'modo_computo', 'tamano_modelo_mb']])
pd.DataFrame(resumen['compute_logs']['tareas']).T[['duracion_s', 'cpu_pct_prom', 'cpu_pct_max', 'ram_mb_max']]

,duracion_s,cpu_pct_prom,cpu_pct_max,ram_mb_prom,ram_mb_max,modo_computo,tamano_modelo_mb
CatBoost_cfg1,29.19,99.51,100.9,348.12,368.16,solo_cpu,0.125
CatBoost_cfg2,58.34,99.93,105.5,350.67,375.05,solo_cpu,0.439
LDA_cfg1,5.29,163.7,371.2,1042.07,1251.98,solo_cpu,0.005
LDA_cfg2,2.7,92.63,100.1,1120.53,1372.61,solo_cpu,0.03
Random_Forest_cfg1,120.9,100.12,100.9,401.75,406.82,solo_cpu,6.845
AdaBoost_cfg1,93.22,100.11,101.8,528.22,534.7,solo_cpu,0.064
AdaBoost_cfg2,212.62,100.13,101.8,640.15,651.94,solo_cpu,0.186
Naive_Bayes_Gaussiano_cfg1,0.52,101.1,101.1,444.59,444.59,solo_cpu,0.004
Naive_Bayes_Gaussiano_cfg2,0.44,101.0,101.0,444.34,444.34,solo_cpu,0.004
MLP_cfg1,17.89,100.21,101.2,1211.3,1228.85,solo_cpu,0.143


,duracion_s,cpu_pct_prom,cpu_pct_max,ram_mb_max
validar_datos,63.09,65.33,101.1,1039.6
preprocesar_datos,73.97,86.19,101.0,2917.97
optimizar_hiperparametros,280.74,1471.14,1589.9,1026.37
entrenar_modelos,275.91,271.69,688.6,4003.07
backtesting,120.52,171.82,1320.3,1986.09
seleccionar_campeon,0.16,12.7,12.7,334.94
registrar_experimentos,50.53,11.94,95.2,424.79
generar_reporte,1.44,66.45,83.9,378.59


In [8]:
# Por que se descarto cada modelo en GetBestModel()
pd.Series(resumen['seleccion']['descartados'], name='motivo').to_frame()

,motivo
